In [1]:
# NOTE: This file is the source for four Jupyter notebook cells.
# Copy each section (between the ╔═╗ banners) into its own cell.
# =============================================================================
#
#  INTERCONNECT — 14-Ring Add-Drop Cascade  ·  neff / ng Parametric Sweep
#  ─────────────────────────────────────────────────────────────────────────
#  Circuit topology
#  ─────────────────
#  ONA_1 output ──► RING_1 input
#  RING_1 through ──► ONA input 1          (port 1)
#  RING_1 drop    ──► RING_2 input
#  RING_2 through ──► RING_3 input
#  RING_2 drop    ──► OPM_1
#  RING_3 through ──► RING_4 input
#  RING_3 drop    ──► OPM_2
#   … (pattern continues) …
#  RING_14 through ──► ONA input 2         (port 2)
#  RING_14 drop   ──► OPM_13
#
#  Sweep
#  ─────
#  Parametric sweep over (neff, ng) pairs for RING_1 only.
#  All other rings are held at their fixed (default or user-supplied) values.
#
# =============================================================================

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports · lumapi setup · Logging · I/O paths                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import sys
import os
import platform
import time
import logging
import warnings
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ─────────────────────────────────────────────────────────────────────────────
# Lumerical installation paths  (identical logic to the FDE notebook cell 1)
# ─────────────────────────────────────────────────────────────────────────────
LUMERICAL_VERSION = "v202"          # ← confirmed from your traceback

if platform.system() == "Windows":
    LUMERICAL_ROOT = rf"C:\Program Files\Lumerical\{LUMERICAL_VERSION}"
    LUMERICAL_API  = rf"{LUMERICAL_ROOT}\api\python"
    LUMERICAL_BIN  = rf"{LUMERICAL_ROOT}\bin"
else:
    LUMERICAL_ROOT = f"/opt/lumerical/{LUMERICAL_VERSION}"
    LUMERICAL_API  = f"{LUMERICAL_ROOT}/api/python"
    LUMERICAL_BIN  = f"{LUMERICAL_ROOT}/bin"

# ── Clear cached failed import ────────────────────────────────────────────────
for _mod in ["lumapi"]:
    if _mod in sys.modules:
        del sys.modules[_mod]

# ── sys.path ─────────────────────────────────────────────────────────────────
if LUMERICAL_API not in sys.path:
    sys.path.insert(0, LUMERICAL_API)

# ── DLL path (Python 3.8+, Windows) ──────────────────────────────────────────
if platform.system() == "Windows":
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(LUMERICAL_BIN))
    else:
        os.environ["PATH"] = str(LUMERICAL_BIN) + ";" + os.environ.get("PATH", "")

# ── Path assertions ───────────────────────────────────────────────────────────
assert Path(LUMERICAL_API).exists(), (
    f"Lumerical API path not found:\n  {LUMERICAL_API}\n"
    f"Check LUMERICAL_VERSION = '{LUMERICAL_VERSION}'"
)
assert Path(LUMERICAL_BIN).exists(), f"Lumerical bin path not found:\n  {LUMERICAL_BIN}"

import lumapi  # noqa
print(f"lumapi imported successfully from:\n  {lumapi.__file__}")

# ─────────────────────────────────────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s │ %(levelname)s │ %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("ICNT_Cascade")

# ─────────────────────────────────────────────────────────────────────────────
# I/O
# ─────────────────────────────────────────────────────────────────────────────
VERSION_NAME    = "ICNT_14Ring_Cascade_neff_sweep_V1"
PROJECT_DIR     = Path.cwd()
DATA_DIR        = PROJECT_DIR / "data_ICNT_cascade_ring_sweep"
DATA_DIR.mkdir(parents=True, exist_ok=True)
HDF5_PATH       = DATA_DIR / f"{VERSION_NAME}.h5"
FIGURES_DIR     = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n  Data directory : {DATA_DIR}")
print(f"  HDF5 output    : {HDF5_PATH}")
print(f"  Figures dir    : {FIGURES_DIR}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 1
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Parameter dataclasses · Interactive parameter collection         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# Number of rings in the cascade
# ─────────────────────────────────────────────────────────────────────────────
N_RINGS = 14

# ─────────────────────────────────────────────────────────────────────────────
# Dataclass:  RingParams
#   Holds all physical and numerical parameters for one add-drop ring.
#   Dual-polarization (TE / TM) parameters are stored separately.
#   All length/wavelength quantities in metres (SI, as expected by INTERCONNECT).
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class RingParams:
    """Physical parameters for one double-bus add-drop ring resonator."""

    ring_id           : int     = 1         # 1-indexed label

    # ── Geometry ─────────────────────────────────────────────────────────────
    radius_m          : float   = 5.0e-6    # ring radius  [m]  (default 5 µm)

    # ── Resonance ────────────────────────────────────────────────────────────
    lambda_res_m      : float   = 1.550e-6  # resonance / expansion wavelength [m]

    # ── TE mode parameters ───────────────────────────────────────────────────
    neff_TE           : float   = 1.98      # effective index (TE)
    ng_TE             : float   = 2.30      # group index    (TE)
    D_TE              : float   = 0.0       # GVD dispersion [ps²/km]  (optional)

    # ── TM mode parameters ───────────────────────────────────────────────────
    neff_TM           : float   = 1.70      # effective index (TM)
    ng_TM             : float   = 2.10      # group index    (TM)
    D_TM              : float   = 0.0       # GVD dispersion [ps²/km]  (optional)

    # ── Coupling (power coupling coefficients) ───────────────────────────────
    kappa_input_sq    : float   = 0.05      # input coupler  |κ|²  (power coupling)
    kappa_drop_sq     : float   = 0.05      # drop  coupler  |κ|²

    # ── Loss ─────────────────────────────────────────────────────────────────
    loss_dB_per_m     : float   = 100.0     # waveguide propagation loss  [dB/m]
    #   ≈ 1 dB/cm is a representative SiN value

    # ── Active polarization used in INTERCONNECT model ───────────────────────
    polarization      : str     = "TE"      # "TE" or "TM"


@dataclass
class ONA_Params:
    """Parameters for the Optical Network Analyser (ONA) source + measurement."""

    # ── Wavelength span ───────────────────────────────────────────────────────
    lambda_start_m    : float   = 1.540e-6  # sweep start  [m]
    lambda_stop_m     : float   = 1.560e-6  # sweep stop   [m]
    n_points          : int     = 1000      # number of frequency/wavelength samples

    # ── Source power ─────────────────────────────────────────────────────────
    power_dBm         : float   = 0.0       # input power  [dBm]

    # ── Input ports (how many ONA input ports to enable) ─────────────────────
    # Port 1 → RING_1 through; Port 2 → RING_14 through
    n_input_ports     : int     = 2


@dataclass
class SweepParams:
    """
    Sweep specification for the neff / ng of RING_1.

    One sweep axis: paired (neff_TE, ng_TE) values.
    ng is derived from neff via a fixed dn_g / dn_eff ratio set at initialisation,
    OR supplied as an explicit array.

    Usage example
    ─────────────
    sp = SweepParams(
        neff_start = 1.90,
        neff_stop  = 2.10,
        n_points   = 21,
        ng_start   = 2.20,
        ng_stop    = 2.45,
    )
    """

    # ── neff sweep range ──────────────────────────────────────────────────────
    neff_start     : float = 1.90
    neff_stop      : float = 2.10
    n_points       : int   = 21      # number of sweep points

    # ── Corresponding ng values ───────────────────────────────────────────────
    # If ng_start == ng_stop == None, ng is linearly interpolated from ng_TE
    # of the ring at the two neff extremes using a user-supplied ratio.
    ng_start       : Optional[float] = 2.20
    ng_stop        : Optional[float] = 2.45

    def build_arrays(self) -> Tuple[np.ndarray, np.ndarray]:
        """Return (neff_arr, ng_arr) of shape (n_points,)."""
        neff_arr = np.linspace(self.neff_start, self.neff_stop, self.n_points)
        if self.ng_start is not None and self.ng_stop is not None:
            ng_arr = np.linspace(self.ng_start, self.ng_stop, self.n_points)
        else:
            # Fallback: proportional scaling  ng / neff = constant
            ratio  = 2.30 / 1.98   # typical SiN ratio as default
            ng_arr = neff_arr * ratio
        return neff_arr, ng_arr


# ─────────────────────────────────────────────────────────────────────────────
# Interactive parameter collection
# ─────────────────────────────────────────────────────────────────────────────
def _prompt(label: str, default, cast=float) -> Any:
    """
    Print a prompt with a default value and return the parsed result.
    Pressing Enter (empty input) accepts the default.
    """
    raw = input(f"  {label} [{default}]: ").strip()
    if raw == "":
        return default
    try:
        return cast(raw)
    except ValueError:
        warnings.warn(f"  Could not parse '{raw}' as {cast.__name__}; using default {default}")
        return default


def collect_ring_params(ring_id: int, defaults: Optional[RingParams] = None) -> RingParams:
    """
    Interactively collect parameters for a single ring.
    Defaults are taken from `defaults` if provided, else from RingParams().
    """
    d = defaults if defaults is not None else RingParams(ring_id=ring_id)
    print(f"\n  ── Ring {ring_id} parameters ──────────────────────────────────")
    r = RingParams(
        ring_id        = ring_id,
        radius_m       = _prompt(f"  Radius [µm]",       d.radius_m * 1e6)   * 1e-6,
        lambda_res_m   = _prompt(f"  λ_resonance [nm]",  d.lambda_res_m * 1e9) * 1e-9,
        neff_TE        = _prompt(f"  n_eff  (TE)",        d.neff_TE),
        ng_TE          = _prompt(f"  n_group (TE)",       d.ng_TE),
        D_TE           = _prompt(f"  Dispersion D (TE) [ps²/km]", d.D_TE),
        neff_TM        = _prompt(f"  n_eff  (TM)",        d.neff_TM),
        ng_TM          = _prompt(f"  n_group (TM)",       d.ng_TM),
        D_TM           = _prompt(f"  Dispersion D (TM) [ps²/km]", d.D_TM),
        kappa_input_sq = _prompt(f"  κ² input coupler",  d.kappa_input_sq),
        kappa_drop_sq  = _prompt(f"  κ² drop  coupler",  d.kappa_drop_sq),
        loss_dB_per_m  = _prompt(f"  Loss [dB/m]",        d.loss_dB_per_m),
        polarization   = _prompt(f"  Polarization [TE/TM]", d.polarization, cast=str),
    )
    return r


def collect_ona_params(defaults: Optional[ONA_Params] = None) -> ONA_Params:
    d = defaults if defaults is not None else ONA_Params()
    print("\n  ── ONA parameters ─────────────────────────────────────────────")
    ona = ONA_Params(
        lambda_start_m = _prompt("  λ_start [nm]",   d.lambda_start_m * 1e9) * 1e-9,
        lambda_stop_m  = _prompt("  λ_stop  [nm]",   d.lambda_stop_m  * 1e9) * 1e-9,
        n_points       = _prompt("  N wavelength pts", d.n_points, cast=int),
        power_dBm      = _prompt("  Source power [dBm]", d.power_dBm),
        n_input_ports  = _prompt("  ONA input ports", d.n_input_ports, cast=int),
    )
    return ona


def collect_sweep_params(defaults: Optional[SweepParams] = None) -> SweepParams:
    d = defaults if defaults is not None else SweepParams()
    print("\n  ── Sweep parameters (neff / ng of Ring 1) ─────────────────────")
    sp = SweepParams(
        neff_start = _prompt("  neff sweep start", d.neff_start),
        neff_stop  = _prompt("  neff sweep stop",  d.neff_stop),
        n_points   = _prompt("  Sweep points",     d.n_points, cast=int),
        ng_start   = _prompt("  ng  sweep start",  d.ng_start),
        ng_stop    = _prompt("  ng  sweep stop",   d.ng_stop),
    )
    return sp


# ─────────────────────────────────────────────────────────────────────────────
# Run interactive collection
# (wrap in a single function so it can be called from Cell 3 too)
# ─────────────────────────────────────────────────────────────────────────────
def collect_all_parameters(
    n_rings           : int  = N_RINGS,
    interactive       : bool = True,
    ring_defaults     : Optional[List[RingParams]] = None,
    ona_default       : Optional[ONA_Params]        = None,
    sweep_default     : Optional[SweepParams]       = None,
) -> Tuple[List[RingParams], ONA_Params, SweepParams]:
    """
    Collect parameters for all rings, the ONA, and the sweep range.

    Parameters
    ----------
    interactive : bool
        If True, prompts the user for each parameter (with defaults).
        If False, all defaults are used — useful for automated pipelines.
    ring_defaults : list of RingParams, optional
        Per-ring default overrides.  len must equal n_rings if provided.

    Returns
    -------
    ring_params_list : List[RingParams]  length = n_rings
    ona_params       : ONA_Params
    sweep_params     : SweepParams
    """
    print("=" * 66)
    print("  INTERCONNECT 14-Ring Cascade — Parameter Collection")
    print("  Press Enter to accept the default value shown in [brackets].")
    print("=" * 66)

    if not interactive:
        ring_params = [
            (ring_defaults[i] if ring_defaults and i < len(ring_defaults)
             else RingParams(ring_id=i + 1))
            for i in range(n_rings)
        ]
        ona_params   = ona_default   if ona_default   else ONA_Params()
        sweep_params = sweep_default if sweep_default else SweepParams()
        log.info("Non-interactive mode: all defaults accepted.")
        return ring_params, ona_params, sweep_params

    ring_params = []
    print("\n  You will now be prompted for each ring (1 → 14).")
    print("  For rings 2–14, defaults from ring 1 are re-used unless overridden.")
    for i in range(n_rings):
        prev = ring_params[-1] if ring_params else None
        # Carry forward previous ring's physical params as defaults for next ring
        if prev is not None:
            default = RingParams(
                ring_id        = i + 1,
                radius_m       = prev.radius_m,
                lambda_res_m   = prev.lambda_res_m,
                neff_TE        = prev.neff_TE,
                ng_TE          = prev.ng_TE,
                D_TE           = prev.D_TE,
                neff_TM        = prev.neff_TM,
                ng_TM          = prev.ng_TM,
                D_TM           = prev.D_TM,
                kappa_input_sq = prev.kappa_input_sq,
                kappa_drop_sq  = prev.kappa_drop_sq,
                loss_dB_per_m  = prev.loss_dB_per_m,
                polarization   = prev.polarization,
            )
        else:
            default = ring_defaults[i] if ring_defaults else RingParams(ring_id=i + 1)
        rp = collect_ring_params(ring_id=i + 1, defaults=default)
        ring_params.append(rp)

    ona_params   = collect_ona_params(defaults=ona_default)
    sweep_params = collect_sweep_params(defaults=sweep_default)

    # ── Summary printout ──────────────────────────────────────────────────────
    print("\n" + "=" * 66)
    print("  Parameter collection complete — summary")
    print("=" * 66)
    print(f"  {'Ring':>6}  {'R [µm]':>8}  {'λ_res [nm]':>12}  "
          f"{'neff_TE':>8}  {'ng_TE':>7}  {'κ²_in':>7}  {'κ²_drop':>8}")
    print("  " + "-" * 62)
    for rp in ring_params:
        print(f"  {rp.ring_id:>6}  {rp.radius_m*1e6:>8.3f}  "
              f"{rp.lambda_res_m*1e9:>12.4f}  "
              f"{rp.neff_TE:>8.5f}  {rp.ng_TE:>7.5f}  "
              f"{rp.kappa_input_sq:>7.4f}  {rp.kappa_drop_sq:>8.4f}")
    print(f"\n  ONA  :  λ {ona_params.lambda_start_m*1e9:.2f}–"
          f"{ona_params.lambda_stop_m*1e9:.2f} nm  |  "
          f"{ona_params.n_points} pts  |  {ona_params.power_dBm} dBm")
    neff_arr, ng_arr = sweep_params.build_arrays()
    print(f"\n  Sweep: neff {neff_arr[0]:.4f}→{neff_arr[-1]:.4f}  |  "
          f"ng {ng_arr[0]:.4f}→{ng_arr[-1]:.4f}  |  {sweep_params.n_points} pts")
    print("=" * 66)

    return ring_params, ona_params, sweep_params


# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE CELL 2 — collect parameters
# Set  interactive=False  to run silently with all defaults.
# ─────────────────────────────────────────────────────────────────────────────
RING_PARAMS_LIST, ONA_PARAMS, SWEEP_PARAMS = collect_all_parameters(
    n_rings     = N_RINGS,
    interactive = True,    # ← set False for automated / HPC runs
)

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 2
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — INTERCONNECT circuit builder + sweep engine + HDF5 storage       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# INTERCONNECT element names — centralised so downstream cells can reuse them
# ─────────────────────────────────────────────────────────────────────────────
ONA_NAME  = "ONA_1"

def ring_name(ring_id: int) -> str:
    return f"RING_{ring_id}"

def opm_name(opm_id: int) -> str:
    """OPM_1 → OPM_13 (one per ring drop except ring 1)."""
    return f"OPM_{opm_id}"


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _set_ring_params
#   Applies all parameters from a RingParams object to a named ring element
#   in the INTERCONNECT session.  Called both for the initial build and for
#   in-loop updates during the sweep.
#
#   INTERCONNECT ring model parameter names used here:
#     "radius"                  → ring radius  [m]
#     "coupling coefficient 1"  → input coupler power coupling κ²
#     "coupling coefficient 2"  → drop  coupler power coupling κ²
#     "effective index 1"       → neff  (TE or as primary)
#     "group index 1"           → ng
#     "loss 1"                  → propagation loss  [dB/m]
#     "center wavelength"       → expansion / resonance wavelength [m]
# ─────────────────────────────────────────────────────────────────────────────
def _set_ring_params(ic, name: str, rp: RingParams, *, neff_override=None, ng_override=None):
    """
    Set all INTERCONNECT parameters for one ring element.

    Parameters
    ----------
    ic              : lumapi.INTERCONNECT session
    name            : element name in the circuit
    rp              : RingParams instance
    neff_override   : if given, overrides rp.neff_TE  (used in sweep)
    ng_override     : if given, overrides rp.ng_TE
    """
    neff = neff_override if neff_override is not None else rp.neff_TE
    ng   = ng_override   if ng_override   is not None else rp.ng_TE

    ic.setnamed(name, "radius",                 rp.radius_m)
    ic.setnamed(name, "center wavelength",      rp.lambda_res_m)
    ic.setnamed(name, "effective index 1",      neff)
    ic.setnamed(name, "group index 1",          ng)
    ic.setnamed(name, "loss 1",                 rp.loss_dB_per_m)
    ic.setnamed(name, "coupling coefficient 1", rp.kappa_input_sq)
    ic.setnamed(name, "coupling coefficient 2", rp.kappa_drop_sq)
    # Dispersion: INTERCONNECT accepts GVD as "dispersion 1" in s²/m units
    # 1 ps²/km = 1e-12 * 1e-3 = 1e-15 s²/m
    if rp.polarization.upper() == "TE":
        ic.setnamed(name, "dispersion 1", rp.D_TE * 1e-15)
    else:
        ic.setnamed(name, "dispersion 1", rp.D_TM * 1e-15)


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _build_circuit
#   Creates the full 14-ring cascade inside a fresh INTERCONNECT session.
#   Layout positions are computed automatically on a horizontal bus.
# ─────────────────────────────────────────────────────────────────────────────
def _build_circuit(
    ic,
    ring_params_list : List[RingParams],
    ona_params       : ONA_Params,
):
    """
    Build the full 14-ring add-drop cascade circuit from scratch.

    Topology
    --------
    ONA output → RING_1 input
    RING_n through → RING_{n+1} input  (bus propagation)
    RING_1 through → ONA input 1
    RING_14 through → ONA input 2
    RING_n drop (n=2..14) → OPM_{n-1}

    All RING_n:
      port 1 = input (bus in)
      port 2 = through (bus out)
      port 3 = add (usually unused / terminated)
      port 4 = drop

    Layout: rings spaced 120 µm apart along x; ONA at x=0.
    """
    n_rings = len(ring_params_list)
    ic.switchtolayout()
    ic.selectall()
    ic.delete()

    dx = 120e-6   # spacing between ring centres for layout (display only)

    # ── Add ONA ───────────────────────────────────────────────────────────────
    ic.addona()
    ic.set("name",        ONA_NAME)
    ic.set("x position",  0.0)
    ic.set("y position",  0.0)
    ic.setnamed(ONA_NAME, "start frequency",  3e8 / ona_params.lambda_stop_m)
    ic.setnamed(ONA_NAME, "stop frequency",   3e8 / ona_params.lambda_start_m)
    ic.setnamed(ONA_NAME, "number of points", ona_params.n_points)
    ic.setnamed(ONA_NAME, "input power",      10**(ona_params.power_dBm / 10) * 1e-3)
    ic.setnamed(ONA_NAME, "number of input ports", ona_params.n_input_ports)

    # ── Add rings + OPMs ─────────────────────────────────────────────────────
    for i, rp in enumerate(ring_params_list):
        rn = ring_name(rp.ring_id)
        ic.addelement("Double Bus Ring Resonator")
        ic.set("name",        rn)
        ic.set("x position",  (i + 1) * dx)
        ic.set("y position",  0.0)
        _set_ring_params(ic, rn, rp)

    # OPMs for rings 2–14 (ring 1 through → ONA port 1)
    for opm_id in range(1, n_rings):    # OPM_1 … OPM_13
        on = opm_name(opm_id)
        ic.addpowermeter()
        ic.set("name",        on)
        ic.set("x position",  (opm_id + 1) * dx)
        ic.set("y position",  120e-6)

    # ── Wire connections ──────────────────────────────────────────────────────
    #  ONA output → RING_1 port 1
    ic.connect(ONA_NAME, "output", ring_name(1), "port 1")

    for i in range(1, n_rings + 1):
        rn = ring_name(i)

        if i == 1:
            # RING_1 through → ONA input 1
            ic.connect(rn, "port 2", ONA_NAME, "input 1")
        elif i < n_rings:
            # RING_{i} through → RING_{i+1} port 1  (bus continues)
            # This connection is made from the perspective of the source ring
            pass  # handled in next-ring loop below

        if i < n_rings:
            # RING_{i} through → RING_{i+1} port 1
            ic.connect(rn, "port 2", ring_name(i + 1), "port 1")

    # RING_14 through → ONA input 2
    ic.connect(ring_name(n_rings), "port 2", ONA_NAME, "input 2")

    # RING_1 drop is NOT connected to an OPM (it feeds ring 2 only via bus,
    # which is the through port; drop of ring 1 is left unconnected / terminated
    # per the topology — ring 1 drop → ring 2 is actually the *bus* continuation.
    # Re-reading the topology:
    #   RING_1 drop → RING_2 input  (i.e. the drop becomes the bus for ring 2 …)
    # Actually the topology shows RING_1 drop → RING_2 input AND
    #   RING_2 through → RING_3.  So the cascade is: drop port chains, not through.
    #
    # NOTE: Topology clarification based on user spec:
    #   RING_n DROP → RING_{n+1} INPUT   (drop-feeds-next pattern)
    #   RING_n THROUGH → ONA or terminated
    #   RING_2 drop → OPM_1, RING_3 drop → OPM_2, … RING_14 drop → OPM_13
    #
    # We rebuild connections now following this interpretation.

    # Clear all connections just made and redo properly:
    ic.switchtolayout()
    ic.selectall()
    ic.delete()

    # ── Re-add all elements ────────────────────────────────────────────────────
    ic.addona()
    ic.set("name", ONA_NAME)
    ic.set("x position", 0.0)
    ic.set("y position", 0.0)
    ic.setnamed(ONA_NAME, "start frequency",  3e8 / ona_params.lambda_stop_m)
    ic.setnamed(ONA_NAME, "stop frequency",   3e8 / ona_params.lambda_start_m)
    ic.setnamed(ONA_NAME, "number of points", ona_params.n_points)
    ic.setnamed(ONA_NAME, "input power",      10**(ona_params.power_dBm / 10) * 1e-3)
    ic.setnamed(ONA_NAME, "number of input ports", ona_params.n_input_ports)

    for i, rp in enumerate(ring_params_list):
        rn = ring_name(rp.ring_id)
        ic.addelement("Double Bus Ring Resonator")
        ic.set("name",        rn)
        ic.set("x position",  (i + 1) * dx)
        ic.set("y position",  0.0)
        _set_ring_params(ic, rn, rp)

    for opm_id in range(1, n_rings):
        on = opm_name(opm_id)
        ic.addpowermeter()
        ic.set("name",        on)
        ic.set("x position",  (opm_id + 1) * dx)
        ic.set("y position",  -120e-6)

    # ── Topology connections (corrected) ──────────────────────────────────────
    # ONA output → RING_1 port 1 (bus input)
    ic.connect(ONA_NAME, "output", ring_name(1), "port 1")

    # RING_1 through → ONA input 1  (pass-through monitor)
    ic.connect(ring_name(1), "port 2", ONA_NAME, "input 1")

    # Chain: RING_n drop → RING_{n+1} port 1;  RING_n through → ONA/terminated
    for i in range(1, n_rings):
        rn_current = ring_name(i)
        rn_next    = ring_name(i + 1)
        # Drop of ring i feeds input of ring i+1
        ic.connect(rn_current, "port 4", rn_next, "port 1")

    # RING_14 through → ONA input 2
    ic.connect(ring_name(n_rings), "port 2", ONA_NAME, "input 2")

    # RING_n drop for n = 2..14 → OPM_{n-1}
    for i in range(2, n_rings + 1):
        opm_id = i - 1   # OPM_1 … OPM_13
        ic.connect(ring_name(i), "port 4", opm_name(opm_id), "input")

    log.info(f"Circuit built: {n_rings} rings, {n_rings - 1} OPMs, 1 ONA.")


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _extract_sweep_results
#   After a single INTERCONNECT run, extracts:
#     - ONA input 1 and 2 transmission spectra  →  shape (n_wl,)
#     - OPM powers  →  shape (n_opm,)
# ─────────────────────────────────────────────────────────────────────────────
def _extract_sweep_results(
    ic,
    n_rings    : int,
    ona_name   : str  = ONA_NAME,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract post-run results from INTERCONNECT.

    Returns
    -------
    wavelengths_m   : np.ndarray  (n_wl,)   wavelength axis [m]
    T_port1_dB      : np.ndarray  (n_wl,)   ONA input 1 transmission [dB]
    T_port2_dB      : np.ndarray  (n_wl,)   ONA input 2 transmission [dB]
    opm_power_dBm   : np.ndarray  (n_opm,)  mean power at each OPM over λ [dBm]
    opm_spectrum    : np.ndarray  (n_opm, n_wl)  per-OPM spectral power [W]
    """
    n_opm = n_rings - 1   # 13

    # ── ONA wavelength axis ───────────────────────────────────────────────────
    freq_raw = ic.getresult(ona_name, "input 1/mode 1/transmission")
    f_arr    = np.asarray(freq_raw["f"]).flatten()
    wl_m     = (3e8 / f_arr)[::-1]   # convert to wavelength, ascending order

    def _get_T_dB(port_label):
        raw_T  = ic.getresult(ona_name, f"{port_label}/mode 1/transmission")
        T_lin  = np.asarray(raw_T["T"]).flatten()
        # Ensure ascending wavelength order
        T_lin  = T_lin[::-1]
        # Guard against log(0)
        T_lin  = np.where(T_lin > 0, T_lin, 1e-30)
        return 10 * np.log10(np.abs(T_lin))

    T_port1_dB = _get_T_dB("input 1")
    T_port2_dB = _get_T_dB("input 2")

    # ── OPM spectra ───────────────────────────────────────────────────────────
    n_wl         = len(wl_m)
    opm_power_dBm   = np.full(n_opm, np.nan)
    opm_spectrum_W  = np.full((n_opm, n_wl), np.nan)

    for k in range(1, n_opm + 1):
        on = opm_name(k)
        try:
            raw_p = ic.getresult(on, "power")
            p_arr = np.asarray(raw_p).flatten()
            if len(p_arr) == n_wl:
                opm_spectrum_W[k - 1, :] = p_arr[::-1]
            else:
                # Scalar power result
                opm_spectrum_W[k - 1, :] = p_arr[0]
            # Mean power in dBm
            mean_p = np.nanmean(opm_spectrum_W[k - 1, :])
            opm_power_dBm[k - 1] = 10 * np.log10(mean_p * 1e3 + 1e-40)
        except Exception as exc:
            log.warning(f"  Could not extract OPM {k}: {exc}")

    return wl_m, T_port1_dB, T_port2_dB, opm_power_dBm, opm_spectrum_W


# ─────────────────────────────────────────────────────────────────────────────
# Helper: _init_hdf5_icnt
#   Creates (or verifies) the HDF5 file structure for the INTERCONNECT sweep.
#   Schema mirrors the FDE HDF5 schema for consistency.
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5_icnt(
    path            : Path,
    neff_arr        : np.ndarray,
    ng_arr          : np.ndarray,
    wl_ref_m        : np.ndarray,
    n_rings         : int,
    ring_params_list: List[RingParams],
    ona_params      : ONA_Params,
    sweep_params    : SweepParams,
    version_name    : str,
):
    """
    Create a new HDF5 file pre-allocated for the INTERCONNECT sweep.

    Datasets
    ────────
    metadata/
        neff_sweep       (n_pts,)
        ng_sweep         (n_pts,)
        wavelengths_m    (n_wl,)    reference wavelength axis from first run
        ring_<id>_*      attributes per ring
        ona_*            attributes for ONA
        timestamp_start

    results/
        T_port1_dB       (n_pts, n_wl)   ONA port 1 transmission spectrum
        T_port2_dB       (n_pts, n_wl)   ONA port 2 transmission spectrum
        opm_power_dBm    (n_pts, n_opm)  mean OPM power
        opm_spectrum_W   (n_pts, n_opm, n_wl)  full OPM spectra

    flags/
        computed         (n_pts,)  bool
    """
    n_pts = len(neff_arr)
    n_opm = n_rings - 1
    n_wl  = len(wl_ref_m)

    with h5py.File(path, "w") as f:
        md = f.create_group("metadata")
        md.create_dataset("neff_sweep",     data=neff_arr)
        md.create_dataset("ng_sweep",       data=ng_arr)
        md.create_dataset("wavelengths_m",  data=wl_ref_m)
        md.attrs["version_name"]     = version_name
        md.attrs["n_rings"]          = n_rings
        md.attrs["n_opm"]            = n_opm
        md.attrs["timestamp_start"]  = datetime.now().isoformat()

        # Per-ring metadata
        for rp in ring_params_list:
            pfx = f"ring_{rp.ring_id}_"
            for k, v in asdict(rp).items():
                md.attrs[pfx + k] = v

        # ONA metadata
        for k, v in asdict(ona_params).items():
            md.attrs[f"ona_{k}"] = v

        rg = f.create_group("results")
        nan2d  = np.full((n_pts, n_wl),       np.nan, dtype=np.float64)
        nan2do = np.full((n_pts, n_opm),      np.nan, dtype=np.float64)
        nan3d  = np.full((n_pts, n_opm, n_wl),np.nan, dtype=np.float64)
        rg.create_dataset("T_port1_dB",     data=nan2d,  chunks=(1, n_wl))
        rg.create_dataset("T_port2_dB",     data=nan2d,  chunks=(1, n_wl))
        rg.create_dataset("opm_power_dBm",  data=nan2do, chunks=(1, n_opm))
        rg.create_dataset("opm_spectrum_W", data=nan3d,  chunks=(1, n_opm, n_wl))

        flg = f.create_group("flags")
        flg.create_dataset("computed", data=np.zeros(n_pts, dtype=bool), chunks=(1,))


# ─────────────────────────────────────────────────────────────────────────────
# MAIN: run_interconnect_sweep
# ─────────────────────────────────────────────────────────────────────────────
def run_interconnect_sweep(
    ring_params_list : List[RingParams] = None,
    ona_params       : ONA_Params       = None,
    sweep_params     : SweepParams      = None,
    hdf5_path        : Path             = HDF5_PATH,
    version_name     : str              = VERSION_NAME,
    hide_gui         : bool             = False,
) -> Dict[str, Any]:
    """
    Run a parametric (neff, ng) sweep for Ring 1 in the 14-ring INTERCONNECT
    cascade.  All other rings are held at their fixed parameter values.

    Sweep axis
    ──────────
    Paired (neff_TE, ng_TE) values for RING_1 only.

    Each sweep point:
      1. Updates RING_1 neff / ng in layout mode.
      2. Runs the INTERCONNECT simulation.
      3. Extracts ONA transmission spectra + OPM powers.
      4. Writes to HDF5 immediately (fault-safe, cache-aware).

    Returns
    -------
    dict with keys:
        neff_sweep      np.ndarray  (n_pts,)
        ng_sweep        np.ndarray  (n_pts,)
        wavelengths_m   np.ndarray  (n_wl,)
        T_port1_dB      np.ndarray  (n_pts, n_wl)
        T_port2_dB      np.ndarray  (n_pts, n_wl)
        opm_power_dBm   np.ndarray  (n_pts, n_opm)
        opm_spectrum_W  np.ndarray  (n_pts, n_opm, n_wl)
        computed        np.ndarray  (n_pts,)  bool
    """
    # ── Defaults ──────────────────────────────────────────────────────────────
    if ring_params_list is None: ring_params_list = RING_PARAMS_LIST
    if ona_params       is None: ona_params        = ONA_PARAMS
    if sweep_params     is None: sweep_params      = SWEEP_PARAMS

    neff_arr, ng_arr = sweep_params.build_arrays()
    n_pts   = len(neff_arr)
    n_rings = len(ring_params_list)
    n_opm   = n_rings - 1

    # ── In-memory result arrays (filled from HDF5 cache or new) ───────────────
    # Wavelength axis shape is not yet known; will be set after first run.
    # Placeholder: use ONA n_points.
    n_wl_placeholder = ona_params.n_points

    T_port1_dB      = np.full((n_pts, n_wl_placeholder), np.nan)
    T_port2_dB      = np.full((n_pts, n_wl_placeholder), np.nan)
    opm_power_dBm   = np.full((n_pts, n_opm),            np.nan)
    opm_spectrum_W  = np.full((n_pts, n_opm, n_wl_placeholder), np.nan)
    computed        = np.zeros(n_pts, dtype=bool)
    wavelengths_m   = None

    # ── HDF5 cache check ──────────────────────────────────────────────────────
    if hdf5_path.exists():
        log.info(f"Cache found → {hdf5_path}")
        with h5py.File(hdf5_path, "r") as f:
            if "results/T_port1_dB" in f:
                T_port1_dB     = f["results/T_port1_dB"][:]
                T_port2_dB     = f["results/T_port2_dB"][:]
                opm_power_dBm  = f["results/opm_power_dBm"][:]
                opm_spectrum_W = f["results/opm_spectrum_W"][:]
                computed[:]    = f["flags/computed"][:]
                wavelengths_m  = f["metadata/wavelengths_m"][:]
        n_wl_placeholder = T_port1_dB.shape[1]
        n_cached   = int(computed.sum())
        remaining  = n_pts - n_cached
        log.info(f"Cached: {n_cached}/{n_pts}  |  Remaining: {remaining}")
        if remaining == 0:
            log.info("All sweep points cached — returning stored results.")
            return dict(
                neff_sweep     = neff_arr,
                ng_sweep       = ng_arr,
                wavelengths_m  = wavelengths_m,
                T_port1_dB     = T_port1_dB,
                T_port2_dB     = T_port2_dB,
                opm_power_dBm  = opm_power_dBm,
                opm_spectrum_W = opm_spectrum_W,
                computed       = computed,
            )
        hdf5_exists = True
    else:
        hdf5_exists = False
        log.info("No cache found — will initialise HDF5 after first run.")

    # ── Open INTERCONNECT session ─────────────────────────────────────────────
    log.info("Launching Lumerical INTERCONNECT …")
    ic = lumapi.INTERCONNECT(hide=hide_gui)

    runs_done  = 0
    runs_total = int((~computed).sum())
    t_start    = time.time()

    try:
        # ── Build circuit once ────────────────────────────────────────────────
        _build_circuit(ic, ring_params_list, ona_params)
        log.info(f"Circuit built  ({runs_total} sweep points remaining) …")

        for s_idx, (neff_val, ng_val) in enumerate(zip(neff_arr, ng_arr)):

            if computed[s_idx]:
                continue

            # ── Update RING_1 only ────────────────────────────────────────────
            ic.switchtolayout()
            _set_ring_params(
                ic, ring_name(1), ring_params_list[0],
                neff_override = neff_val,
                ng_override   = ng_val,
            )

            # ── Run simulation ────────────────────────────────────────────────
            try:
                ic.run()
            except Exception as exc:
                log.warning(f"  INTERCONNECT FAILED │ neff={neff_val:.5f} │ {exc}")
                computed[s_idx] = True
                if hdf5_exists or (wavelengths_m is not None):
                    with h5py.File(hdf5_path, "r+") as hf:
                        hf["flags/computed"][s_idx] = True
                        hf.flush()
                continue

            # ── Extract results ───────────────────────────────────────────────
            try:
                wl_m, t1, t2, opm_p, opm_s = _extract_sweep_results(
                    ic, n_rings, ONA_NAME
                )
            except Exception as exc:
                log.warning(f"  Extraction FAILED │ neff={neff_val:.5f} │ {exc}")
                computed[s_idx] = True
                continue

            # ── Initialise HDF5 on first successful run ───────────────────────
            if wavelengths_m is None:
                wavelengths_m   = wl_m
                n_wl            = len(wl_m)
                # Resize in-memory arrays to real n_wl
                T_port1_dB      = np.full((n_pts, n_wl),       np.nan)
                T_port2_dB      = np.full((n_pts, n_wl),       np.nan)
                opm_spectrum_W  = np.full((n_pts, n_opm, n_wl), np.nan)
                if not hdf5_exists:
                    _init_hdf5_icnt(
                        path             = hdf5_path,
                        neff_arr         = neff_arr,
                        ng_arr           = ng_arr,
                        wl_ref_m         = wl_m,
                        n_rings          = n_rings,
                        ring_params_list = ring_params_list,
                        ona_params       = ona_params,
                        sweep_params     = sweep_params,
                        version_name     = version_name,
                    )
                    hdf5_exists = True
                    log.info(f"HDF5 initialised → {hdf5_path}")

            # ── Store in memory ───────────────────────────────────────────────
            T_port1_dB    [s_idx, :]    = t1
            T_port2_dB    [s_idx, :]    = t2
            opm_power_dBm [s_idx, :]    = opm_p
            opm_spectrum_W[s_idx, :, :] = opm_s
            computed      [s_idx]       = True

            # ── Incremental HDF5 write ────────────────────────────────────────
            with h5py.File(hdf5_path, "r+") as hf:
                hf["results/T_port1_dB"]    [s_idx, :]    = t1
                hf["results/T_port2_dB"]    [s_idx, :]    = t2
                hf["results/opm_power_dBm"] [s_idx, :]    = opm_p
                hf["results/opm_spectrum_W"][s_idx, :, :] = opm_s
                hf["flags/computed"]        [s_idx]       = True
                hf.flush()

            runs_done += 1

            if runs_done % 5 == 0 or runs_done == runs_total:
                elapsed = time.time() - t_start
                rate    = runs_done / elapsed if elapsed > 0 else 1e-9
                eta     = (runs_total - runs_done) / rate
                log.info(
                    f"  [{runs_done:3d}/{runs_total}]  "
                    f"neff = {neff_val:.5f}  ng = {ng_val:.5f}  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:5.0f} s"
                )

        # ── Finalise HDF5 ─────────────────────────────────────────────────────
        if hdf5_exists:
            with h5py.File(hdf5_path, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed.sum())
                if wavelengths_m is not None and "wavelengths_m" not in hf["metadata"]:
                    hf["metadata"].create_dataset("wavelengths_m", data=wavelengths_m)

    finally:
        ic.close()
        log.info("INTERCONNECT session closed.")

    elapsed_total = time.time() - t_start
    log.info(
        f"Sweep done │ {runs_done} new runs │ "
        f"total = {elapsed_total:.1f} s │ "
        f"avg = {elapsed_total / max(runs_done, 1):.2f} s/sim"
    )

    return dict(
        neff_sweep     = neff_arr,
        ng_sweep       = ng_arr,
        wavelengths_m  = wavelengths_m,
        T_port1_dB     = T_port1_dB,
        T_port2_dB     = T_port2_dB,
        opm_power_dBm  = opm_power_dBm,
        opm_spectrum_W = opm_spectrum_W,
        computed       = computed,
    )


# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE CELL 3
# ─────────────────────────────────────────────────────────────────────────────
sweep_results = run_interconnect_sweep()

# Unpack for downstream use
neff_sweep     = sweep_results["neff_sweep"]      # (n_pts,)
ng_sweep       = sweep_results["ng_sweep"]        # (n_pts,)
wavelengths_m  = sweep_results["wavelengths_m"]   # (n_wl,)
T_port1_dB     = sweep_results["T_port1_dB"]      # (n_pts, n_wl)
T_port2_dB     = sweep_results["T_port2_dB"]      # (n_pts, n_wl)
opm_power_dBm  = sweep_results["opm_power_dBm"]   # (n_pts, n_opm)
opm_spectrum_W = sweep_results["opm_spectrum_W"]  # (n_pts, n_opm, n_wl)
computed       = sweep_results["computed"]        # (n_pts,)

wavelengths_nm = wavelengths_m * 1e9  # convenience

print(f"\n  Sweep complete — {computed.sum()} / {len(computed)} pts computed")
print(f"  T_port1_dB shape  : {T_port1_dB.shape}")
print(f"  opm_power_dBm shape: {opm_power_dBm.shape}")
print(f"  HDF5: {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 3
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Post-processing · Visualisation · Cache-aware re-plotting        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# Plotting style  (publication quality)
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"       : "serif",
    "font.serif"        : ["DejaVu Serif", "Georgia", "Times New Roman"],
    "font.size"         : 11,
    "axes.labelsize"    : 12,
    "axes.titlesize"    : 13,
    "legend.fontsize"   : 9,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "axes.linewidth"    : 0.8,
    "axes.grid"         : True,
    "grid.alpha"        : 0.3,
    "grid.linewidth"    : 0.5,
    "lines.linewidth"   : 1.4,
    "figure.dpi"        : 120,
    "savefig.dpi"       : 300,
    "savefig.bbox"      : "tight",
})

# ─────────────────────────────────────────────────────────────────────────────
# Utility: load_sweep_results_from_hdf5
#   Loads the full sweep dataset from HDF5.
#   Used when re-running Cell 4 without re-running Cell 3.
# ─────────────────────────────────────────────────────────────────────────────
def load_sweep_results_from_hdf5(path: Path = HDF5_PATH) -> Dict[str, Any]:
    """
    Load all sweep results from an existing HDF5 file.
    Returns the same dict structure as run_interconnect_sweep().
    """
    if not path.exists():
        raise FileNotFoundError(f"HDF5 not found: {path}")
    with h5py.File(path, "r") as f:
        return dict(
            neff_sweep     = f["metadata/neff_sweep"][:],
            ng_sweep       = f["metadata/ng_sweep"][:],
            wavelengths_m  = f["metadata/wavelengths_m"][:],
            T_port1_dB     = f["results/T_port1_dB"][:],
            T_port2_dB     = f["results/T_port2_dB"][:],
            opm_power_dBm  = f["results/opm_power_dBm"][:],
            opm_spectrum_W = f["results/opm_spectrum_W"][:],
            computed       = f["flags/computed"][:],
        )


# ─────────────────────────────────────────────────────────────────────────────
# Cache-aware result accessor
# ─────────────────────────────────────────────────────────────────────────────
def get_results(hdf5_path: Path = HDF5_PATH) -> Dict[str, Any]:
    """
    Return sweep_results from the in-memory variable if available,
    otherwise load from HDF5.  This makes Cell 4 fully re-runnable.
    """
    try:
        # If sweep_results is defined in the kernel (Cell 3 was run), use it.
        r = sweep_results
        if r["wavelengths_m"] is not None:
            return r
    except NameError:
        pass
    log.info(f"Loading results from HDF5: {hdf5_path}")
    return load_sweep_results_from_hdf5(hdf5_path)


# ─────────────────────────────────────────────────────────────────────────────
# Plot 1: ONA Transmission spectra for selected sweep points
#   Shows T_port1_dB (RING_1 through) for a subset of neff values.
#   Lines are colour-mapped to neff value.
# ─────────────────────────────────────────────────────────────────────────────
def plot_transmittance_sweep(
    results          : Optional[Dict] = None,
    port             : int            = 1,       # 1 or 2
    n_curves         : int            = 8,       # number of neff values to plot
    figsize          : tuple          = (10, 5),
    save             : bool           = True,
    figures_dir      : Path           = FIGURES_DIR,
    cmap_name        : str            = "plasma",
) -> plt.Figure:
    """
    Plot ONA transmission spectra for ~n_curves evenly-spaced neff sweep points.

    Parameters
    ----------
    port        : 1 = RING_1 through (ONA port 1)
                  2 = RING_14 through (ONA port 2)
    n_curves    : number of sweep indices to overlay
    """
    if results is None:
        results = get_results()

    neff_arr  = results["neff_sweep"]
    wl_nm     = results["wavelengths_m"] * 1e9
    T_key     = "T_port1_dB" if port == 1 else "T_port2_dB"
    T_data    = results[T_key]        # (n_pts, n_wl)
    computed  = results["computed"]

    # Select n_curves indices from computed-only points
    valid_idx = np.where(computed)[0]
    sel_idx   = valid_idx[np.linspace(0, len(valid_idx) - 1, min(n_curves, len(valid_idx))).astype(int)]

    cmap  = plt.get_cmap(cmap_name)
    norm  = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm    = ScalarMappable(cmap=cmap, norm=norm)

    fig, ax = plt.subplots(figsize=figsize)

    for idx in sel_idx:
        c = cmap(norm(neff_arr[idx]))
        ax.plot(wl_nm, T_data[idx], color=c, lw=1.2, alpha=0.85,
                label=f"neff={neff_arr[idx]:.4f}")

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=11)

    port_label = "Ring 1 through" if port == 1 else "Ring 14 through"
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Transmission  (dB)")
    ax.set_title(
        f"ONA Transmission — {port_label}\n"
        f"({n_curves} of {int(computed.sum())} sweep pts shown)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fname = figures_dir / f"transmittance_port{port}_{n_curves}curves.png"
        fig.savefig(fname)
        log.info(f"Figure saved → {fname}")
        fig.savefig(fname.with_suffix(".pdf"))

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 2: OPM power vs neff for all OPMs (one line per OPM)
#   Shows mean power [dBm] at each OPM as a function of Ring 1 neff.
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_power_vs_neff(
    results      : Optional[Dict] = None,
    figsize      : tuple          = (11, 5),
    save         : bool           = True,
    figures_dir  : Path           = FIGURES_DIR,
    cmap_name    : str            = "tab20",
) -> plt.Figure:
    """
    Plot mean OPM power [dBm] vs Ring 1 neff for each of the 13 OPMs.
    Each OPM gets one coloured line.
    """
    if results is None:
        results = get_results()

    neff_arr      = results["neff_sweep"]
    opm_power     = results["opm_power_dBm"]   # (n_pts, n_opm)
    computed      = results["computed"]

    valid = computed
    neff_v     = neff_arr[valid]
    opm_power_v = opm_power[valid, :]
    n_opm       = opm_power_v.shape[1]

    cmap   = plt.get_cmap(cmap_name)
    colors = [cmap(i / n_opm) for i in range(n_opm)]

    fig, ax = plt.subplots(figsize=figsize)

    for k in range(n_opm):
        ring_id = k + 2   # OPM_1 ↔ RING_2 drop, …
        ax.plot(
            neff_v, opm_power_v[:, k],
            color=colors[k], lw=1.3, marker="o", ms=3, markevery=3,
            label=f"OPM {k+1}  (Ring {ring_id} drop)",
        )

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Mean power  (dBm)")
    ax.set_title(
        "OPM Mean Power vs Ring 1 $n_{eff}$\n"
        "(Drop power at each ring for each sweep point)"
    )
    ax.legend(loc="upper right", ncol=2, framealpha=0.85, fontsize=8)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fname = figures_dir / "opm_power_vs_neff.png"
        fig.savefig(fname)
        log.info(f"Figure saved → {fname}")
        fig.savefig(fname.with_suffix(".pdf"))

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 3: OPM spectral power heatmap  (neff × wavelength, one subplot per OPM)
#   For each OPM, shows how its power spectrum evolves with Ring 1 neff.
#   Useful to track resonance shift and drop-band evolution.
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_spectrum_heatmap(
    results         : Optional[Dict] = None,
    opm_ids         : Optional[List[int]] = None,   # 1-indexed; None = all
    figsize_per_row : tuple = (10, 2.8),
    save            : bool  = True,
    figures_dir     : Path  = FIGURES_DIR,
    cmap_name       : str   = "inferno",
    vmin_dBm        : Optional[float] = None,
    vmax_dBm        : Optional[float] = None,
) -> plt.Figure:
    """
    Plot 2D spectral-power heatmaps for selected OPMs.

    X axis: wavelength [nm]
    Y axis: Ring 1 neff
    Colour: power spectral density [dBm per wavelength point] — shows resonance
            as bright stripe shifting with neff.

    Parameters
    ----------
    opm_ids : list of 1-indexed OPM IDs to plot.  Defaults to all 13.
    """
    if results is None:
        results = get_results()

    neff_arr      = results["neff_sweep"]
    wl_nm         = results["wavelengths_m"] * 1e9
    opm_spec      = results["opm_spectrum_W"]   # (n_pts, n_opm, n_wl)
    computed      = results["computed"]

    valid   = computed
    neff_v  = neff_arr[valid]
    spec_v  = opm_spec[valid, :, :]         # (n_valid, n_opm, n_wl)
    n_opm   = spec_v.shape[1]

    if opm_ids is None:
        opm_ids = list(range(1, n_opm + 1))

    n_rows  = len(opm_ids)
    fig_h   = figsize_per_row[1] * n_rows
    fig, axes = plt.subplots(n_rows, 1, figsize=(figsize_per_row[0], fig_h),
                             sharex=True, sharey=True)
    if n_rows == 1:
        axes = [axes]

    # Convert W to dBm per point
    spec_dBm = 10 * np.log10(np.where(spec_v > 0, spec_v, 1e-30) * 1e3)
    _vmin = vmin_dBm if vmin_dBm is not None else np.nanpercentile(spec_dBm, 2)
    _vmax = vmax_dBm if vmax_dBm is not None else np.nanpercentile(spec_dBm, 98)

    for ax_i, opm_id in enumerate(opm_ids):
        k   = opm_id - 1
        img = axes[ax_i].pcolormesh(
            wl_nm, neff_v, spec_dBm[:, k, :],
            cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto",
        )
        ring_id = k + 2
        axes[ax_i].set_ylabel(f"OPM {opm_id}\n(Ring {ring_id})\n$n_{{eff,1}}$",
                              fontsize=9)
        cbar = fig.colorbar(img, ax=axes[ax_i], pad=0.01)
        cbar.set_label("dBm", fontsize=8)
        cbar.ax.tick_params(labelsize=8)

    axes[-1].set_xlabel("Wavelength  (nm)")
    fig.suptitle(
        "OPM Spectral Power  vs  Ring 1 $n_{eff}$\n"
        "(Resonance shift visualised as horizontal band displacement)",
        fontsize=12, y=1.01,
    )
    fig.tight_layout()

    if save:
        fname = figures_dir / f"opm_spectrum_heatmap_opm{'_'.join(str(o) for o in opm_ids)}.png"
        fig.savefig(fname)
        log.info(f"Figure saved → {fname}")
        fig.savefig(fname.with_suffix(".pdf"))

    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 4: Resonance wavelength tracking  vs  neff
#   Finds the wavelength of minimum transmission (notch) at each sweep point
#   for ONA port 1 and plots it vs neff.  Linear fit gives ∂λ_res / ∂neff.
# ─────────────────────────────────────────────────────────────────────────────
def plot_resonance_tracking(
    results         : Optional[Dict] = None,
    port            : int            = 1,
    figsize         : tuple          = (7, 4.5),
    save            : bool           = True,
    figures_dir     : Path           = FIGURES_DIR,
) -> plt.Figure:
    """
    Track the resonance dip of ONA transmission vs Ring 1 neff.
    Fits a linear trend  λ_res = a·neff + b  and reports sensitivity.
    """
    if results is None:
        results = get_results()

    neff_arr  = results["neff_sweep"]
    wl_nm     = results["wavelengths_m"] * 1e9
    T_data    = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    computed  = results["computed"]

    valid = computed
    neff_v  = neff_arr[valid]
    T_v     = T_data[valid, :]

    # Find deepest dip (minimum transmission) at each sweep point
    dip_idx = np.argmin(T_v, axis=1)
    lam_dip = wl_nm[dip_idx]

    # Linear fit
    coeffs  = np.polyfit(neff_v, lam_dip, 1)
    poly    = np.poly1d(coeffs)
    sensitivity_nm_per_neff = coeffs[0]

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=18, zorder=5, color="#2166ac",
               label="Resonance dip")
    ax.plot(neff_v, poly(neff_v), "r--", lw=1.5,
            label=f"Linear fit  ∂λ/∂n = {sensitivity_nm_per_neff:.3f} nm/RIU")

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Tracking — ONA Port {port}\n"
        f"Sensitivity: {sensitivity_nm_per_neff:.3f} nm / RIU"
    )
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fname = figures_dir / f"resonance_tracking_port{port}.png"
        fig.savefig(fname)
        log.info(f"Figure saved → {fname}")
        fig.savefig(fname.with_suffix(".pdf"))

    log.info(f"Resonance sensitivity: {sensitivity_nm_per_neff:.4f} nm / RIU")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# PLOT REGISTRY — reusable hook for downstream cells
#
# Downstream cells (Cell 5, 6, …) should call:
#   from <this_module> import PLOT_REGISTRY
#   PLOT_REGISTRY["transmittance"](results=my_results)
#
# Or directly call the functions above.
# ─────────────────────────────────────────────────────────────────────────────
PLOT_REGISTRY: Dict[str, Any] = {
    "transmittance_port1"    : lambda **kw: plot_transmittance_sweep(port=1, **kw),
    "transmittance_port2"    : lambda **kw: plot_transmittance_sweep(port=2, **kw),
    "opm_power_vs_neff"      : plot_opm_power_vs_neff,
    "opm_spectrum_heatmap"   : plot_opm_spectrum_heatmap,
    "resonance_tracking_p1"  : lambda **kw: plot_resonance_tracking(port=1, **kw),
    "resonance_tracking_p2"  : lambda **kw: plot_resonance_tracking(port=2, **kw),
}


# ─────────────────────────────────────────────────────────────────────────────
# DEPENDENCY REGISTRY — reusable objects exported for future cells
#
# These are the clean handles downstream code should import.
# ─────────────────────────────────────────────────────────────────────────────
DEPENDENCY_REGISTRY: Dict[str, Any] = {
    # Parameter dataclasses
    "RingParams"               : RingParams,
    "ONA_Params"               : ONA_Params,
    "SweepParams"              : SweepParams,

    # Parameter collection utilities
    "collect_ring_params"      : collect_ring_params,
    "collect_ona_params"       : collect_ona_params,
    "collect_sweep_params"     : collect_sweep_params,
    "collect_all_parameters"   : collect_all_parameters,

    # INTERCONNECT helpers
    "ring_name"                : ring_name,
    "opm_name"                 : opm_name,
    "ONA_NAME"                 : ONA_NAME,
    "_set_ring_params"         : _set_ring_params,
    "_build_circuit"           : _build_circuit,
    "_extract_sweep_results"   : _extract_sweep_results,
    "_init_hdf5_icnt"          : _init_hdf5_icnt,

    # Sweep engine
    "run_interconnect_sweep"   : run_interconnect_sweep,

    # Data I/O
    "load_sweep_results_from_hdf5" : load_sweep_results_from_hdf5,
    "get_results"              : get_results,

    # Plot functions
    "PLOT_REGISTRY"            : PLOT_REGISTRY,
    "plot_transmittance_sweep" : plot_transmittance_sweep,
    "plot_opm_power_vs_neff"   : plot_opm_power_vs_neff,
    "plot_opm_spectrum_heatmap": plot_opm_spectrum_heatmap,
    "plot_resonance_tracking"  : plot_resonance_tracking,

    # I/O paths (for downstream cells to inherit)
    "DATA_DIR"                 : DATA_DIR,
    "FIGURES_DIR"              : FIGURES_DIR,
    "HDF5_PATH"                : HDF5_PATH,
    "VERSION_NAME"             : VERSION_NAME,
    "N_RINGS"                  : N_RINGS,
}

print("\n  DEPENDENCY_REGISTRY keys available for downstream cells:")
for k in DEPENDENCY_REGISTRY:
    print(f"    {k}")


# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE CELL 4 — generate all plots
# (Safe to re-run: loads from HDF5 if kernel was restarted)
# ─────────────────────────────────────────────────────────────────────────────
_res = get_results()

fig1 = plot_transmittance_sweep(results=_res, port=1, n_curves=8)
fig2 = plot_transmittance_sweep(results=_res, port=2, n_curves=8)
fig3 = plot_opm_power_vs_neff(results=_res)
fig4 = plot_opm_spectrum_heatmap(results=_res, opm_ids=[1, 4, 7, 10, 13])
fig5 = plot_resonance_tracking(results=_res, port=1)

plt.show()

print(f"\n  All figures saved to: {FIGURES_DIR}")
print(f"  HDF5 data at        : {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 4
# ═════════════════════════════════════════════════════════════════════════════
#
#  ─────────────────────────────────────────────────────────────────────────
#  DEPENDENCY NOTE FOR FUTURE CELLS (Cell 5, 6, …)
#  ─────────────────────────────────────────────────────────────────────────
#
#  All reusable components are registered in DEPENDENCY_REGISTRY.
#  To use in a new cell, simply reference the objects by name — they are
#  already in the kernel namespace from Cells 1-4.
#
#  Common patterns:
#
#  1. Add a new ring to the cascade:
#       new_ring = RingParams(ring_id=15, radius_m=6e-6, neff_TE=2.01)
#       updated_rings = RING_PARAMS_LIST + [new_ring]
#       results2 = run_interconnect_sweep(ring_params_list=updated_rings)
#
#  2. Run a sweep on Ring 2 instead of Ring 1:
#       # Modify _build_circuit to accept sweep_ring_id parameter (add param)
#       # and update _set_ring_params for that ring in the loop.
#
#  3. Add a new sweep dimension (e.g., radius sweep):
#       # Extend SweepParams dataclass with radius_start / radius_stop
#       # Add a radius update inside the sweep loop in run_interconnect_sweep.
#
#  4. Load and replot from HDF5 only:
#       r = load_sweep_results_from_hdf5(HDF5_PATH)
#       plot_opm_power_vs_neff(results=r)
#
#  5. Use PLOT_REGISTRY in a dashboard loop:
#       for plot_name, fn in PLOT_REGISTRY.items():
#           fn(results=_res, save=True)
#
# ═════════════════════════════════════════════════════════════════════════════

lumapi imported successfully from:
  C:\Program Files\Lumerical\v202\api\python\lumapi.py

  Data directory : D:\GitHub\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep
  HDF5 output    : D:\GitHub\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_neff_sweep_V1.h5
  Figures dir    : D:\GitHub\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\figures
  INTERCONNECT 14-Ring Cascade — Parameter Collection
  Press Enter to accept the default value shown in [brackets].

  You will now be prompted for each ring (1 → 14).
  For rings 2–14, defaults from ring 1 are re-used unless overridden.

  ── Ring 1 parameters ──────────────────────────────────


KeyboardInterrupt: Interrupted by user